# 3D Fluorescence LIVE/DEAD Segmentation

This notebook is a **generic template** for 3D fluorescence image segmentation and quantification of LIVE/DEAD-stained samples.

**Before running the workflow:**
- Replace the example file path, ROI, experiment name, and channel definitions in **Section 3** with your own data.
- Make sure the channel names in `stain_dict` match the metadata printed after loading your file.
- Start with the default settings and tune only the parameters that are necessary for your dataset.

---

# 1. Import Required Libraries
This cell imports all necessary libraries for image processing and visualization.

In [ ]:
#[Cell 1]
# Import all required libraries for image processing and visualization.
from helpers.notebook_setup_helpers import load_ld_notebook_setup
globals().update(load_ld_notebook_setup())

#### Helper Functions



The reusable helper functions for this notebook are stored in `helpers/notebook_helpers.py`.

This keeps the notebook lighter and makes the workflow sections easier to read.

If you update the helper file, rerun the next code cell to reload the functions.


In [ ]:
#[Cell 2]
# Reload helper functions from the external module.
# Rerun this cell after editing helpers/notebook_helpers.py.
from helpers.notebook_helpers import reload_helpers
reload_helpers()

# 3. Inputs and Setup

Replace the example values in the following cells with information from **your own dataset** before running the analysis.

In [ ]:
#[Cell 3]

# Replace `input_file` with your own microscopy file.
# Supported examples include `.nd2` and multi-channel `.tif/.tiff` files.
input_file = 'path/to/your_image_file.nd2'

# Set to True to load a selected ROI from large datasets.
# Set to False to load the full stack directly.
big_image = True

# ROI format: [x0, x1, y0, y1, z0, z1]
# Use 0 for any limit you want to keep unchanged.
# Example: ROI = [100, 900, 150, 1200, 0, 40]
ROI = [0, 0, 0, 0, 0, 0]

# This prefix is used when saving setup files and output tables.
name_setup = 'your_experiment_name'

# Approximate diameters used during segmentation (in micrometers).
nuclei_diameter = 10  # um, approximate cell diameter for LIVE/DEAD-stained cells
cell_diameter = 30    # um, used only for threshold window sizing

# Keep these values at 1.0 unless you intentionally want to apply an extra manual scale correction.
scale_factor = 1.0
zoom_factors = [1.0, 1.0, 1.0]  # [Z, Y, X]

# Reuse previously saved napari contrast/gamma settings if the CSV file exists.
use_setup = True

# Set to True to use the StarDist model instead of watershed-based segmentation.
trig_stardist = False

# Set to True to compute multi-marker combinations in the quantification summary.
multilabel = True

# Nucleus/cell splitting configuration (used even for L/D union segmentation).
# Profile options: "balanced", "conservative", "aggressive"
nuclei_split_config = get_nuclei_split_config(profile="conservative")

# z_split_aggressive: controls how z-stack consistency is enforced after watershed.
#   - False (default): topology-only — labels are split only when their voxels
#     form truly disconnected 3D regions across the z-stack. Fast and conservative.
#   - True: watershed-reinforced — if a label has multiple significant 2D components
#     at ANY z-slice, a local watershed carves it along the constriction, even when
#     the components reconnect at other z-levels. Slower but more aggressive.
nuclei_split_config["z_split_aggressive"] = False

In [ ]:
#[Cell 4]
# Load image, read metadata, and compute derived parameters.
(meta, img, r_X, r_Y, r_Z, file_meta, ROI_print,
 cyto_factor, PCM_factor, zoom_factors) = initialize_dataset(
    input_file, ROI, big_image,
    nuclei_diameter=nuclei_diameter,
    cell_diameter=cell_diameter,
    scale_factor=scale_factor,
    zoom_factors=zoom_factors,
)

In [ ]:
#[Cell 5]
# Define the channels in your dataset.
# Format: 'CONDITION_NAME': ['Marker label', 'channel name from file metadata', 'display color']
# Note: do NOT add a NUCLEI entry for LIVE/DEAD assays; all cells are segmented together.
stain_dict = {
    'LIVE': ['CALCEIN', 'CHANNEL_LIVE', 'Green'],
    'DEAD': ['ETHD', 'CHANNEL_DEAD', 'Red'],
}

# 4. Information

This section calculates derived image parameters (cell dimensions, volumes), builds the staining table from your inputs, and opens an interactive Napari viewer for contrast and gamma setup.

In [ ]:
#[Cell 6]
# Prepare image stack, build stain table, and open napari channel preview.
(im_final_stack, nuclei_radius, cell_radius, nuclei_volume, cell_volume,
 stain_df, viewer_0) = prepare_and_preview(
    img, meta, ROI, big_image,
    nuclei_diameter, cell_diameter,
    stain_dict, file_meta,
    napari_module=napari,
)

### Acquisition processing setup

In [ ]:
#[Cell 7]
# Load or create napari contrast/gamma settings for each channel.
stain_df, stain_complete_df, original_stain_complete_df = prepare_stain_settings(
    im_final_stack['Original image'],
    stain_df=stain_df,
    name_setup=name_setup,
    use_setup=use_setup,
    settings=settings,
    napari_module=napari,
)

In [ ]:
#[Cell 8]
# Display the complete stain settings table for review.
stain_complete_df

# 5. Image Processing and Segmentation

This section normalizes the image, resamples to isotropic voxels, denoises, applies contrast and gamma adjustments, and then segments the structures of interest.

In [ ]:
#[Cell 9]
# Normalize each channel independently to [0, 255].
im_final_stack['Normalized image'] = normalize_image_channels(im_final_stack['Original image'])
hist_plot(im_final_stack['Normalized image'], stain_complete_df)

In [ ]:
#[Cell 10]
# Resample the image to isotropic voxel spacing.
im_final_stack['Zoomed image'], r_zX, r_zY, r_zZ = resample_to_isotropic(
    im_final_stack['Normalized image'],
    zoom_factors,
    meta=meta
)
hist_plot(im_final_stack['Zoomed image'], stain_complete_df)

In [ ]:
#[Cell 11]
# Apply median filter for noise removal.
im_final_stack['Denoised image'] = apply_median_denoise(im_final_stack['Zoomed image'])
hist_plot(im_final_stack['Denoised image'], stain_complete_df)

In [ ]:
#[Cell 12]
# Apply per-channel contrast and gamma adjustments from stain settings.
im_final_stack['Adjusted image'] = apply_contrast_gamma_per_channel(
    im_final_stack['Denoised image'],
    stain_complete_df
)
hist_plot(im_final_stack['Adjusted image'], stain_complete_df)

In [ ]:
#[Cell 13]
# --- User input ---
# sigma: standard deviation (in voxels) for the Gaussian smoothing kernel.
sigma = 0.5

# Apply Gaussian smoothing.
im_final_stack['Filtered image'] = apply_gaussian_smoothing(
    im_final_stack['Adjusted image'],
    sigma
)
hist_plot(im_final_stack['Filtered image'], stain_complete_df)

In [ ]:
#[Cell 14]
# --- User input ---
num_plateaus = 2
plateau_factor = 0.7

# Apply histogram equalization to support thresholding.
im_final_stack['Equalized image'] = apply_histogram_equalization_per_channel(
    im_final_stack['Filtered image'],
    num_plateaus,
    plateau_factor
)
hist_plot(im_final_stack['Equalized image'], stain_complete_df)

In [ ]:
#[Cell 15]
# --- User input ---
threshold_method = 'otsu'

# Apply combined thresholding (global + Sauvola local + statistical background + gain).
im_final_stack["Threshold image"] = apply_threshold_per_channel(
    im_final_stack["Equalized image"],
    stain_complete_df=stain_complete_df,
    nuclei_diameter=nuclei_diameter,
    cell_diameter=cell_diameter,
    r_zxyz=(r_zX, r_zY, r_zZ),
    threshold_method=threshold_method,
)

In [ ]:
#[Cell 16]
# Export histograms for every processing stage and a Parameters sheet.
output_path = Path(input_file).stem + '_histograms.xlsx'

processing_params = {
    'Input file':                     str(input_file),
    'ROI [x0,x1,y0,y1,z0,z1]':        str(ROI_print),
    'Big image mode':                 str(big_image),
    'Experiment name':                str(name_setup),
    'Nuclei diameter (um)':           str(nuclei_diameter),
    'Cell diameter (um)':             str(cell_diameter),
    'Zoom factors [Z,Y,X]':           str(zoom_factors),
    'Scale factor':                   str(scale_factor),
    'Use StarDist':                   str(trig_stardist),
    'Multilabel':                     str(multilabel),
    'Nuclei split config':            str(nuclei_split_config),
    'Sigma (Gaussian smoothing)':     str(sigma),
    'Num plateaus (histogram eq.)':   str(num_plateaus),
    'Plateau factor (histogram eq.)': str(plateau_factor),
    'Threshold method':               str(threshold_method),
}

export_channel_histograms(
    im_final_stack,
    stain_complete_df,
    output_path,
    processing_params=processing_params,
)
print(f"Saved to: {output_path}")

## Segmentation

In [ ]:
#[Cell 17]
# Segment cells from the combined LIVE+DEAD binary mask using union-labeling.
im_segmentation_stack = segment_nuclei(
    im_final_stack,
    stain_df=stain_df,
    stain_complete_df=stain_complete_df,
    nuclei_split_config=nuclei_split_config,
    r_zxyz=(r_zX, r_zY, r_zZ),
    nuclei_diameter=nuclei_diameter,
    trig_stardist=trig_stardist,
)

# Add a virtual NUCLEI entry so downstream helpers can reference it.
if 'NUCLEI' not in stain_complete_df.index:
    stain_complete_df.loc['NUCLEI'] = ['', '', '', '', '', '']

In [ ]:
#[Cell 18]
# Assign segmented cell labels to each LIVE/DEAD channel.
im_segmentation_stack = assign_channel_labels(
    im_final_stack,
    im_segmentation_stack=im_segmentation_stack,
    stain_df=stain_df,
)

In [ ]:
#[Cell 19]
# Open napari viewers showing all processing stages and segmentation results.
viewer_0, viewer_1 = view_processing_results(
    im_final_stack,
    im_segmentation_stack=im_segmentation_stack,
    stain_df=stain_df,
    stain_complete_df=stain_complete_df,
    r_xyz=(r_X, r_Y, r_Z),
    r_zxyz=(r_zX, r_zY, r_zZ),
    napari_module=napari,
)

# 6. Quantification and Analysis

This section quantifies nuclei and cell properties, computes statistics, and visualizes distributions. Results are exported for further analysis.

In [ ]:
#[Cell 20]
# Quantify marker overlap and intensity per segmented object.
filtered_img = im_final_stack['Filtered image']
r_xyz = (r_zX, r_zY, r_zZ)

labels_dict = build_labels_dict(
    im_segmentation_stack,
    filtered_img,
    stain_complete_df=stain_complete_df,
    stain_df=stain_df,
    r_xyz=r_xyz,
    zooms=zoom_factors,
    multilabel=multilabel,
)

In [ ]:
#[Cell 21]
# Convert per-object dict to a tidy DataFrame.
labels_df, truncated_df = labels_dict_to_dataframe(
    labels_dict,
    truncate=True,
)

In [ ]:
#[Cell 22]
# Preview labels DataFrame.
truncated_df

In [ ]:
#[Cell 23]
# Print population-level summary: LIVE/DEAD cell counts and percentages.
print_population_summary(
    labels_df,
    stain_complete_df=stain_complete_df,
    stain_df=stain_df,
)

In [ ]:
#[Cell 24]
# Build the full (untruncated) quantification table at original resolution for export.
labels_full_dict = build_full_labels_dict(
    im_segmentation_stack,
    im_final_stack,
    filtered_img,
    stain_complete_df=stain_complete_df,
    r_xyz=(r_X, r_Y, r_Z),
    zooms=zoom_factors,
)

In [ ]:
#[Cell 25]
# Convert full dict to DataFrame for export.
labels_full_df = labels_dict_to_dataframe(labels_full_dict)

## Evaluate cell distribution in the space

In [ ]:
#[Cell 26]
# Plot 2D and 3D spatial distributions of LIVE/DEAD cells.
im_in = im_final_stack['Filtered image']
plot_spatial_distributions(
    labels_df,
    stain_complete_df=stain_complete_df,
    stain_df=stain_df,
    im_in=im_in,
    r_X=r_X, r_Y=r_Y, r_Z=r_Z,
    zoom_factors=zoom_factors,
)

## Evaluate cell size distribution

In [ ]:
#[Cell 27]
# Plot size distributions (volume, diameter) for each LIVE/DEAD population.
plot_size_distributions(
    labels_df,
    stain_complete_df=stain_complete_df,
    stain_df=stain_df,
)

## Create .VTK volume

In [ ]:
#[Cell 28]
# Build labelled VTK volume meshes for each segmented LIVE/DEAD population.
build_vtk_volumes(
    im_segmentation_stack,
    labels_full_df=labels_full_df,
    stain_complete_df=stain_complete_df,
    input_file=input_file,
    r_xyz=(r_X, r_Y, r_Z),
    zoom_factors=zoom_factors,
)

## Create .STL for markers

In [ ]:
#[Cell 29]
# Export per-marker binary volumes as STL mesh files.
export_marker_stl(
    im_segmentation_stack,
    stain_df=stain_df,
    stain_complete_df=stain_complete_df,
    input_file=input_file,
)

### Create a complete report XSL

In [ ]:
#[Cell 30]
# Export full quantification results, stain settings, and parameters to Excel.
export_quantification_to_excel(
    output_path,
    original_stain_complete_df,
    labels_full_df,
)

# CREATE .inp FOR FINITE ELEMENT ANALYSIS

In [ ]:
#[Cell 31]
# Generate an Abaqus .inp mesh from the segmentation for finite element analysis.
export_fea_mesh(
    im_segmentation_stack,
    input_file=input_file,
)